# Path and Query Parameters

This notebook covers:

1. Capture path parameters and convert types automatically
2. Capture query parameters with defaults and constraints
3. Validate parameters with `Query()` and `Path()` metadata
4. Document parameters via OpenAPI without extra work

**Scope**: FastAPI + `TestClient`. Each section spins up its own tiny app.

We stay in the portfolio domain — `/assets/{ticker}`, `/orders/{order_id}`, `?asset_class=equity` — so the patterns map directly into the capstone.

## 1. Path Parameters

Anything inside `{...}` in the route path becomes a function argument. The type hint drives both **parsing** (`"42"` → `42`) and **validation** (422 if it doesn't parse).

Order matters: more specific routes must come before more generic ones, otherwise `/assets/active` will be captured by `/assets/{ticker}` as `ticker="active"`.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/assets/active")
def list_active():
    return {"active": ["AAPL", "MSFT"]}

@app.get("/assets/{ticker}")
def get_asset(ticker: str):
    return {"ticker": ticker}

@app.get("/orders/{order_id}")
def get_order(order_id: int):
    return {"order_id": order_id, "type": type(order_id).__name__}

client = TestClient(app)

print("GET /assets/active   ->", client.get("/assets/active").json())
print("GET /assets/AAPL     ->", client.get("/assets/AAPL").json())
print("GET /orders/42       ->", client.get("/orders/42").json())
print("GET /orders/not-int  ->", client.get("/orders/not-int").status_code,
      client.get("/orders/not-int").json()["detail"][0]["msg"])

## 2. Query Parameters

Any handler argument **not** in the path becomes a query parameter. The function default determines whether it's required.

- No default → required (`limit: int`). Missing query string param = 422.
- With default → optional (`limit: int = 10`).
- Default of `None` + `| None` → optional, may be absent (`q: str | None = None`).

In [ ]:
app = FastAPI()

@app.get("/assets")
def list_assets(limit: int = 10, offset: int = 0, q: str | None = None):
    return {"limit": limit, "offset": offset, "q": q}

client = TestClient(app)

print("default values         ->", client.get("/assets").json())
print("?limit=5&offset=20     ->", client.get("/assets", params={"limit": 5, "offset": 20}).json())
print("?q=apple               ->", client.get("/assets", params={"q": "apple"}).json())

# Type coercion still applies — '5' (string) becomes 5 (int).
print("?limit=abc (bad)       ->", client.get("/assets", params={"limit": "abc"}).status_code,
      client.get("/assets", params={"limit": "abc"}).json()["detail"][0]["msg"])

## 3. Defaults vs Required

The pattern looks innocuous but trips a lot of people. The three flavors:

| Signature                  | Behavior                                  |
|----------------------------|-------------------------------------------|
| `limit: int`               | Required. Missing → 422.                  |
| `limit: int = 10`          | Optional, defaults to 10.                 |
| `limit: int \| None = None` | Optional, may be `None`.                  |

Use the third form when "absent" is a meaningful state distinct from "use the default" — e.g., a search query that, when unset, means "don't filter.

In [ ]:
app = FastAPI()

@app.get("/required")
def require(limit: int):  # no default → required
    return {"limit": limit}

@app.get("/optional-with-default")
def with_default(limit: int = 10):
    return {"limit": limit}

@app.get("/optional-nullable")
def nullable(q: str | None = None):
    return {"q": q, "filtered": q is not None}

client = TestClient(app)

print("/required (no param)        ->", client.get("/required").status_code,
      client.get("/required").json()["detail"][0]["msg"])
print("/required?limit=3           ->", client.get("/required", params={"limit": 3}).json())
print("/optional-with-default      ->", client.get("/optional-with-default").json())
print("/optional-nullable          ->", client.get("/optional-nullable").json())
print("/optional-nullable?q=apple  ->", client.get("/optional-nullable", params={"q": "apple"}).json())

## 4. `Query(...)` and `Path(...)` Metadata

`Query()` and `Path()` attach metadata: validation, OpenAPI documentation, examples, aliases. They behave like Pydantic's `Field()` but for non-body params.

Common knobs:

- `ge`, `le`, `gt`, `lt` — numeric bounds.
- `min_length`, `max_length`, `pattern` — string constraints.
- `alias` — when the URL has a name you can't use as a Python identifier (e.g., `page-size`).
- `description`, `examples` — surfaced in `/docs`.

In [ ]:
from fastapi import Query, Path
from typing import Annotated

app = FastAPI()

@app.get("/assets/{ticker}")
def get_asset(
    ticker: Annotated[str, Path(min_length=1, max_length=10, pattern=r"^[A-Z.]+$", description="Uppercase ticker")],
    page_size: Annotated[int, Query(ge=1, le=100, alias="page-size", description="Page size, 1-100")] = 20,
):
    return {"ticker": ticker, "page_size": page_size}

client = TestClient(app)

print("GET /assets/AAPL                   ->", client.get("/assets/AAPL").json())
print("GET /assets/AAPL?page-size=50      ->", client.get("/assets/AAPL", params={"page-size": 50}).json())

# Constraint violations -> 422
print("GET /assets/aapl (lowercase)       ->", client.get("/assets/aapl").status_code)
print("GET /assets/AAPL?page-size=500     ->", client.get("/assets/AAPL", params={"page-size": 500}).status_code)
print("GET /assets/AAPL?page-size=0       ->", client.get("/assets/AAPL", params={"page-size": 0}).status_code)

# The constraints leak into OpenAPI for free — find them in the schema.
schema = client.get("/openapi.json").json()
print("\nopenapi parameters for /assets/{ticker}:")
for p in schema["paths"]["/assets/{ticker}"]["get"]["parameters"]:
    print(" ", p["name"], "in", p["in"], "schema:", p["schema"])

## 5. Multiple Values: `list[int]` Query Params

Repeating a key in the query string sends multiple values. Type the argument as `list[T]` and FastAPI parses them.

This is the right shape for `?ids=1&ids=2&ids=3` style filters. The comma-separated form (`?ids=1,2,3`) is *not* an HTTP standard — many clients and proxies mangle it. Stick to repeated keys.

In [ ]:
app = FastAPI()

@app.get("/assets")
def list_assets(
    ids: Annotated[list[int], Query(description="Repeat the param to send multiple IDs")] = [],
):
    return {"ids": ids, "count": len(ids)}

client = TestClient(app)

print("GET /assets                  ->", client.get("/assets").json())
print("GET /assets?ids=1&ids=2&ids=3 ->", client.get("/assets?ids=1&ids=2&ids=3").json())
print("GET /assets?ids=abc (bad)    ->", client.get("/assets?ids=abc").status_code)

## 6. Enums for Constrained Values

When a parameter has a fixed set of legal values, an `Enum` is better than a free-form string + manual check:

- Pydantic rejects unknown values with a 422.
- The enum members appear in the OpenAPI schema as a `enum` array.
- Swagger UI renders them as a dropdown.
- Refactors are safer because the type system enforces the set.

For string enums to JSON-serialize correctly, subclass `str, Enum`.

In [ ]:
from enum import Enum

class AssetClass(str, Enum):
    equity = "equity"
    bond = "bond"
    cash = "cash"

app = FastAPI()

@app.get("/assets")
def list_assets(asset_class: AssetClass | None = None):
    return {"asset_class": asset_class}

client = TestClient(app)

print("GET /assets                       ->", client.get("/assets").json())
print("GET /assets?asset_class=equity    ->", client.get("/assets", params={"asset_class": "equity"}).json())
print("GET /assets?asset_class=crypto    ->", client.get("/assets", params={"asset_class": "crypto"}).status_code,
      client.get("/assets", params={"asset_class": "crypto"}).json()["detail"][0]["msg"])

# The enum members surface in the OpenAPI schema as `enum`.
schema = client.get("/openapi.json").json()
print("\nasset_class param schema:")
for p in schema["paths"]["/assets"]["get"]["parameters"]:
    if p["name"] == "asset_class":
        print(" ", p["schema"])

## Key Takeaways

- **Path params** come from `{...}` in the route; type hints drive parsing and validation.
- **Query params** are any handler arg not in the path. The default determines required vs optional.
- **`Path()` / `Query()` metadata** add constraints (`ge`, `le`, `min_length`, `pattern`, `alias`) and documentation; they show up in OpenAPI automatically.
- **`list[T]`** parses repeated query keys (`?ids=1&ids=2`). Avoid comma-separated — not in the HTTP spec.
- **Enums** are the right shape for fixed value sets: validation + docs + IDE autocomplete in one declaration.

Next: notebook 2.2 — bodies, response shaping, and dropping back to raw `Response` when you need to.

## Exercises

**1. Enum filter.** Add an `asset_class: AssetClass | None = None` query param to a `/assets` endpoint. When set, filter an in-memory list of assets to that class; when unset, return all.

**2. Paginate with metadata.** Build `GET /assets` accepting:
- `page: int` — defaults to 1, must be ≥ 1.
- `page_size: int` — defaults to 20, must be in `[1, 100]`, aliased as `page-size`.

Return `{"page": ..., "page_size": ..., "items": [...]}` with a sliced view of an in-memory list. Use `Query(...)` metadata so the constraints appear in `/openapi.json`.

**3. Multi-ticker lookup.** Build `GET /assets/lookup?ticker=AAPL&ticker=MSFT&ticker=GOOGL` that accepts a `list[str]` of tickers and returns the matching subset of an in-memory dict. Reject empty lists with a 422 (hint: `Query(min_length=1)`).